In [1]:
import pandas as pd
import pyreadstat
from pathlib import Path
from typing import List

In [ ]:
cols = [
    'wt_final',
    'wt_time',
    'Reg9',
    'xStrata',
    'Relig7',
    'LondInOut',
    'LA_2023',
    'Age9',
    'nchild',
    'nadult',
    'Disab3',
    'Gend3',
    'Educ6',
    'Orient4',
    'disty1_POP',
    'disty10_POP',
    'disty11_POP',
    'disty12_POP',
    'disty13_POP',
    'disty2_POP',
    'disty3_POP',
    'disty4_POP',
    'disty5_POP',
    'disty6_POP',
    'disty7_POP',
    'disty8_POP',
    'disty9_POP',
    'Eth2',
    'Eth7',
    'health',
    'NSSEC4',
    'NSSEC5',
    'NSSEC8',
    'IMD10',
    'IMD10_GR2',
    'IMD10_GR3',
    'IMD10_GR5',
    'IMD4',
    'WorkStat10',
    'WorkStat5',
    'WorkStat7',
    'Equalities_Metric_2024_GR2',
    'BMIG',
    'DVBMI',
    'DVHeightM',
    'DVWeightKG',
    'fruitveg',
    'MEMS7_ALL',
    'MEMS7GR_ALL',
    'MEMS7_SPORTCOUNT_A01',
    'MEMS7_OUT_SPORTCOUNT_A01',
    'MEMS7_IN_SPORTCOUNT_A01',
    'DURATION_SPORTCOUNT_A01',
    'VolAny',
    'VolCnt',
    'VolFrqB_Pop',
    'volint1',
    'volint2',
    'volint3',
    'volint4',
    'volint5',
    'volint6',
    'volint7',
    'Motiva_POP',
    'motivb_POP',
    'motivc_POP',
    'motivd_POP',
    'motive_POP',
    'motivex2a',
    'motivex2b',
    'motivex2c',
    'motivex2d',
    'inclus_a',
    'inclus_b',
    'inclus_c',
    'anxious',
    'comm1',
    'comm2',
    'happy',
    'indev',
    'indevtry',
    'lifesat',
    'lone',
    'worthw',
    'workactlvl',
    'WHOWITHA_SPORTCOUNT_A01',
    'WHOWITHB_SPORTCOUNT_A01',
    'WHOWITHC_SPORTCOUNT_A01',
    'WHOWITHD_SPORTCOUNT_A01',
    'limfreti1',
    'limfreti2',
    'limfreti3',
    'limfreti4',
    'limfreti5',
    'limfreti6',
    'limfreti7',
    'limfreti8',
    'MONTHS_12_FOOTBALL_F01',
    'MONTHS_12_CRICKET_F02',
    'MONTHS_12_RUGBYUNION_F03',
    'MONTHS_12_RUGBYLEAGUE_F04',
    'MONTHS_12_WHEELCHRUGBY_F05',
    'MONTHS_12_NETBALL_F06',
    'MONTHS_12_BASKETBALL_F07',
    'MONTHS_12_WHEELCHBASKETBALL_F08',
    'MONTHS_12_HOCKEY_F09',
    'MONTHS_12_VOLLEYBALL_F10',
    'MONTHS_12_ROUNDERS_F11',
    'MONTHS_12_DODGEBALL_F12',
    'MONTHS_12_BASESOFTBALL_F13',
    'MONTHS_12_LACROSSE_F14',
    'MONTHS_12_GOALBALL_F15',
    'MONTHS_12_HANDBALL_F16',
    ## Added
    'CULFRQ_1_9_POP'
    'CULFRQ_1_9_POP2'
    'CULMTH_1_9_POP'
]


In [3]:
len(cols)
print(cols[-1])

MONTHS_12_HANDBALL_F16


In [4]:
def query_col(col_name : str, meta : pyreadstat.metadata_container) -> str:
    names = meta.column_names
    idx = names.index(col_name)
    return meta.column_labels[idx]

def get_df(path: Path, cols):
    if 'LondInOut' not in cols:
        cols.append('LondInOut')
    df , meta = pyreadstat.read_sav(path, usecols = cols)
    df = df[df['LondInOut'].notna()]
    return df, meta

def retrieve_clean(cols : List[str], data_path :str, table_dict : dict) -> pd.DataFrame:
    master_df = []
    verify_check = verify_paths(data_path, table_dict)
    if len(verify_check) > 0:
        raise FileNotFoundError(f'FilesNotFound:\n{verify_check}')
    print('All Files Found!')
    for k, v in table_dict.items():

        data_pathf = data_path.format(
            a = k,
            b = v['start'],
            c = v['end'],
            d = v['range'],
            e = v['year_num'],
            f = v['fid']
        )
        full_path = Path(data_pathf)

        df, meta = get_df(full_path, cols)

        for i in cols:
            if i not in df.columns:
                print(f'Column not found in {v['start']} - {v['end']} df: {i}')
        df['year'] = f'{v['start']} - {v['end']}'
        master_df.append(df)
        print(f'Retrieved data for: {v['start']} - {v['end']}')
    combined = pd.concat(master_df)
    return combined, meta

def verify_paths(path : str, table_dict : dict) -> list:
    filesnotfound = []
    for k, v in table_dict.items():
        pathf = path.format(
            a = k,
            b = v['start'],
            c = v['end'],
            d = v['range'],
            e = v['year_num'],
            f = v['fid']
        )
  
        outcome = Path(pathf).exists()
        if outcome == False:
            filesnotfound.append(pathf)
        continue
    return filesnotfound

table_dict = {
    9288 : {
        'start' : 2022,
        'end' : 2023,
        'range' : '22-23',
        'year_num' : 'year_8',
        'fid' : '20250103'

    },
    9136 : {
        'start' : 2021,
        'end' : 2022,
        'range' : '21-22',
        'year_num' : 'year_7',
        'fid' : '20250103'
    },
    8993 : {
        'start' : 2020,
        'end': 2021,
        'range' : '20-21',
        'year_num' : 'year_6',
        'fid' : '20250103'
    },
    8899 : {
        'start' : 2019,
        'end': 2020,
        'range' : '19-20',
        'year_num' : 'year_5',
        'fid' : '20250103'
    },
    8652 : {
        'start' : 2018,
        'end': 2019,
        'range': '18-19',
        'year_num' : 'year_4',
        'fid' : '20250103'
    },
    8651 : {
        'start' : 2017,
        'end' : 2018,
        'range' : '17-18',
        'year_num' : 'year_3',
        'fid' : '20250103'
    },
    8391 : {
        'start' : 2016,
        'end' : 2017,
        'range' : '16-17',
        'year_num' : 'year_2',
        'fid' : '20250106'
    },
    8223 : {
        'start' : 2015,
        'end' : 2016, 
        'range' : '15-16',
        'year_num' : 'year_1',
        'fid' : '20250106'
    }
}
master_path = 'C:/Masters/London Sport/{a}_ActiveLifeSurvey_{b}_{c}/UKDA-{a}-spss/spss/spss28/active_lives_survey_nov_{d}_data_{e}_shared_{f}.sav'  

In [5]:
df, meta = retrieve_clean(cols, master_path, table_dict)
df.to_csv('master.csv', index=False)

All Files Found!
Retrieved data for: 2022 - 2023
Column not found in 2021 - 2022 df: inclus_a
Column not found in 2021 - 2022 df: inclus_b
Column not found in 2021 - 2022 df: inclus_c
Column not found in 2021 - 2022 df: workactlvl
Column not found in 2021 - 2022 df: limfreti1
Column not found in 2021 - 2022 df: limfreti2
Column not found in 2021 - 2022 df: limfreti3
Column not found in 2021 - 2022 df: limfreti4
Column not found in 2021 - 2022 df: limfreti5
Column not found in 2021 - 2022 df: limfreti6
Column not found in 2021 - 2022 df: limfreti7
Column not found in 2021 - 2022 df: limfreti8
Retrieved data for: 2021 - 2022
Column not found in 2020 - 2021 df: inclus_a
Column not found in 2020 - 2021 df: inclus_b
Column not found in 2020 - 2021 df: inclus_c
Column not found in 2020 - 2021 df: workactlvl
Column not found in 2020 - 2021 df: limfreti1
Column not found in 2020 - 2021 df: limfreti2
Column not found in 2020 - 2021 df: limfreti3
Column not found in 2020 - 2021 df: limfreti4
Col

In [ ]:
cols = ['Gend3', 'Disab3', 'Age5', 'Relig7', 'Orient4', 'Educ6', 'Eth7', 'DVBMI', 'VolAny', 'NSSEC5', 'DURATION_SPORTCOUNT_A01', 'MEMS7_IN_SPORTCOUNT_A01', 'MEMS7_OUT_SPORTCOUNT_A01']


In [6]:
len(df.columns)

114

In [7]:
path = r'C:\Users\fergu\Documents\GitHub\london_sport2\exploration\exploration_data\master_csv.csv'
df = pd.read_csv(path)


In [8]:
len(cols)

113

In [9]:
x = df.isnull().sum().to_frame()

In [10]:
pd.set_option('display.max_rows', None)
print(df.isnull().sum())

wt_final                           0
wt_time                          588
xStrata                            0
Reg9                               0
LondInOut                          0
LA_2023                            0
Age9                            1341
nadult                         72433
nchild                         71805
Disab3                          9229
disty1_POP                      3200
disty10_POP                     3200
disty11_POP                     3200
disty12_POP                     3200
disty13_POP                     3200
disty2_POP                      3200
disty3_POP                      3200
disty4_POP                      3200
disty5_POP                      3200
disty6_POP                      3200
disty7_POP                      3200
disty8_POP                      3200
disty9_POP                      3200
Educ6                           8844
Eth2                            9293
Eth7                            9293
Gend3                            303
h

In [11]:
cleaned = df.dropna()
len(cleaned)

155